# 04 · Flood Susceptibility Model Training

**Step 4** — train and evaluate interpretable classifiers (Logistic Regression, Random Forest, optional XGBoost) to predict per-cell flood occurrence.

We report ROC-AUC, F1, accuracy, confusion matrix, ROC curve and feature importance, then persist the best model for the scoring engine and dashboard.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
from src import modeling, utils
from src.feature_engineering import INDICATOR_NAMES
from src.utils import PROCESSED_DIR

try:
    table = pd.read_parquet(PROCESSED_DIR / 'feature_table.parquet')
except Exception:
    table = pd.read_csv(PROCESSED_DIR / 'feature_table.csv')
print('Feature table:', table.shape)

In [ ]:
out = modeling.train_and_evaluate(table)
summary = pd.DataFrame({k: v['metrics'] for k, v in out['results'].items()}).T
summary.round(3)

## ROC, confusion matrix & feature importance (interactive Plotly)

In [ ]:
from src import visualization as viz
best = out['results'][out['best_model_name']]
viz.roc_figure(out['results']).show()
viz.confusion_figure(best['confusion_matrix']).show()
viz.feature_importance_figure(best['feature_importance']).show()

In [ ]:
modeling.save_model(out['best_model'], out['best_model_name'], out['feature_columns'])
indicators = {n: np.load(PROCESSED_DIR / f'indicator_{n}.npy') for n in INDICATOR_NAMES}
surface = modeling.predict_surface(out['best_model'], indicators, out['feature_columns'])
np.save(PROCESSED_DIR / 'model_surface.npy', surface)
print('Best model:', out['best_model_name'], '| saved model + probability surface')
print('Next: 05_risk_scoring.ipynb')